<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/performance_comparison_expansion_divisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series – Performance Comparison: expansion_divisor Impact

![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)

This notebook demonstrates the performance impact of using the `expansion_divisor` parameter in [OptiPFair](https://github.com/peremartra/optipfair) when pruning transformer models.

We compare models pruned to the same expansion rate (2.6x or 260%) with and without `expansion_divisor=256`, measuring:
- **TTFT** (Time To First Token): Latency before generation starts
- **Tokens/Second**: Generation throughput

## Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (T4 or better recommended)  
- **Dependencies**: Installed automatically in the first cell

## by Pere Martra

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!

---

If you want your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

# Performance Comparison: expansion_divisor Impact

This notebook compares the inference performance of models pruned with different `expansion_divisor` settings.

**Test Scenarios:**
1. **Base Model**: Original unpruned model (baseline)
2. **Pruned (no divisor)**: Pruned to 260% expansion rate, no rounding
3. **Pruned (divisor=256)**: Pruned to 260% expansion rate, rounded to multiple of 256

**Metrics:**
- **TTFT (Time To First Token)**: How long before generation starts
- **Tokens/Second**: Generation throughput
- **Model Size**: Parameter count comparison

Author: Pere Martra

Designed for Google Colab - GPU runtime recommended

---
## Installation and Setup

In [ ]:
!pip install -q transformers optipfair torch pandas matplotlib seaborn

## Import Libraries and Check GPU

In [ ]:
import torch
import gc
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from optipfair import prune_model

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected. Performance comparison may not be accurate on CPU.")

## Configuration

In [ ]:
# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Pruning configuration
TARGET_EXPANSION_RATE = 260  # 2.6x expansion rate (260%)

# Benchmark configuration
TEST_PROMPTS = [
    "The theory of relativity states that",
    "In the field of artificial intelligence,",
    "Paris is the capital of",
    "Machine learning algorithms can",
    "The human brain consists of",
]

MAX_NEW_TOKENS = 50  # Tokens to generate per prompt
NUM_WARMUP_RUNS = 2  # Warmup iterations (not measured)
NUM_MEASURED_RUNS = 5  # Measured iterations per prompt

print("Configuration set successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Target expansion rate: {TARGET_EXPANSION_RATE}%")
print(f"Test prompts: {len(TEST_PROMPTS)}")
print(f"Tokens per generation: {MAX_NEW_TOKENS}")
print(f"Warmup runs: {NUM_WARMUP_RUNS}")
print(f"Measured runs: {NUM_MEASURED_RUNS}")

## Utility Functions

In [ ]:
def count_parameters(model):
    """Count total parameters in model"""
    return sum(p.numel() for p in model.parameters())

def get_intermediate_size(model):
    """Get the intermediate size from the first MLP layer"""
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        return model.model.layers[0].mlp.gate_proj.out_features
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        return model.transformer.h[0].mlp.c_fc.out_features
    return None

def cleanup_memory():
    """Clean up GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def measure_inference_time(model, tokenizer, prompt, max_new_tokens=50, num_runs=5, warmup_runs=2):
    """
    Measure TTFT and tokens/second for a given model and prompt.
    
    Returns:
        dict with 'ttft', 'tokens_per_second', 'total_time', 'generated_tokens'
    """
    model.eval()
    
    # Prepare input
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_length = inputs['input_ids'].shape[1]
    
    ttft_times = []
    total_times = []
    generated_tokens_list = []
    
    # Warmup runs
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False,
            )
        cleanup_memory()
    
    # Measured runs
    for _ in range(num_runs):
        start_time = time.time()
        
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False,
            )
        
        end_time = time.time()
        
        # Calculate metrics
        generated_length = outputs.shape[1]
        num_generated_tokens = generated_length - input_length
        total_time = end_time - start_time
        
        # Approximate TTFT (this is simplified - first token time)
        # In practice, TTFT ≈ total_time / num_generated_tokens for small batches
        ttft = total_time / num_generated_tokens
        
        ttft_times.append(ttft)
        total_times.append(total_time)
        generated_tokens_list.append(num_generated_tokens)
        
        cleanup_memory()
    
    # Calculate averages
    avg_ttft = sum(ttft_times) / len(ttft_times)
    avg_total_time = sum(total_times) / len(total_times)
    avg_generated_tokens = sum(generated_tokens_list) / len(generated_tokens_list)
    tokens_per_second = avg_generated_tokens / avg_total_time
    
    return {
        'ttft': avg_ttft,
        'tokens_per_second': tokens_per_second,
        'total_time': avg_total_time,
        'generated_tokens': avg_generated_tokens
    }

def benchmark_model(model, tokenizer, prompts, max_new_tokens=50, num_runs=5, warmup_runs=2):
    """
    Benchmark a model across multiple prompts.
    
    Returns:
        dict with aggregated metrics
    """
    all_ttft = []
    all_tps = []
    all_total_time = []
    
    for prompt in prompts:
        metrics = measure_inference_time(
            model, tokenizer, prompt, 
            max_new_tokens=max_new_tokens,
            num_runs=num_runs,
            warmup_runs=warmup_runs
        )
        all_ttft.append(metrics['ttft'])
        all_tps.append(metrics['tokens_per_second'])
        all_total_time.append(metrics['total_time'])
    
    return {
        'avg_ttft': sum(all_ttft) / len(all_ttft),
        'avg_tokens_per_second': sum(all_tps) / len(all_tps),
        'avg_total_time': sum(all_total_time) / len(all_total_time),
        'ttft_std': pd.Series(all_ttft).std(),
        'tps_std': pd.Series(all_tps).std(),
    }

print("Utility functions defined successfully!")

## Load and Benchmark Base Model

First, let's load the original model and establish our baseline performance.

In [ ]:
print(f"Loading base model: {MODEL_NAME}")
print("="*60)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model info
base_params = count_parameters(base_model)
base_intermediate = get_intermediate_size(base_model)
base_hidden = base_model.config.hidden_size

print(f"\nBase Model Information:")
print(f"  Total parameters: {base_params:,}")
print(f"  Intermediate size: {base_intermediate:,}")
print(f"  Hidden size: {base_hidden:,}")
print(f"  Expansion rate: {(base_intermediate/base_hidden)*100:.2f}%")
print(f"  Divisible by 256: {base_intermediate % 256 == 0}")

print("\n" + "="*60)
print("Benchmarking base model...")
print("="*60)

base_metrics = benchmark_model(
    base_model, 
    tokenizer, 
    TEST_PROMPTS,
    max_new_tokens=MAX_NEW_TOKENS,
    num_runs=NUM_MEASURED_RUNS,
    warmup_runs=NUM_WARMUP_RUNS
)

print(f"\nBase Model Performance:")
print(f"  Avg TTFT: {base_metrics['avg_ttft']*1000:.2f} ms (±{base_metrics['ttft_std']*1000:.2f})")
print(f"  Avg Tokens/Second: {base_metrics['avg_tokens_per_second']:.2f} (±{base_metrics['tps_std']:.2f})")
print(f"  Avg Total Time: {base_metrics['avg_total_time']:.3f} s")

## Create Pruned Model (No Divisor)

Prune to 260% expansion rate without using `expansion_divisor`.

In [ ]:
print("\n" + "="*60)
print("Creating pruned model WITHOUT expansion_divisor")
print("="*60)

# Clean up and reload model
del base_model
cleanup_memory()

model_no_divisor = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

# Prune without divisor
pruned_no_divisor, stats_no_divisor = prune_model(
    model=model_no_divisor,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",
    pruning_percentage=None,  # Must be None when using expansion_rate
    expansion_rate=TARGET_EXPANSION_RATE,
    expansion_divisor=None,  # No rounding
    show_progress=True,
    return_stats=True
)

# Model info
no_div_params = count_parameters(pruned_no_divisor)
no_div_intermediate = get_intermediate_size(pruned_no_divisor)

print(f"\nPruned Model (No Divisor) Information:")
print(f"  Total parameters: {no_div_params:,}")
print(f"  Intermediate size: {no_div_intermediate:,}")
print(f"  Expansion rate: {stats_no_divisor['expansion_rate']:.2f}%")
print(f"  Parameter reduction: {stats_no_divisor['percentage_reduction']:.2f}%")
print(f"  Divisible by 256: {no_div_intermediate % 256 == 0}")

print("\n" + "="*60)
print("Benchmarking pruned model (no divisor)...")
print("="*60)

no_div_metrics = benchmark_model(
    pruned_no_divisor, 
    tokenizer, 
    TEST_PROMPTS,
    max_new_tokens=MAX_NEW_TOKENS,
    num_runs=NUM_MEASURED_RUNS,
    warmup_runs=NUM_WARMUP_RUNS
)

print(f"\nPruned Model (No Divisor) Performance:")
print(f"  Avg TTFT: {no_div_metrics['avg_ttft']*1000:.2f} ms (±{no_div_metrics['ttft_std']*1000:.2f})")
print(f"  Avg Tokens/Second: {no_div_metrics['avg_tokens_per_second']:.2f} (±{no_div_metrics['tps_std']:.2f})")
print(f"  Avg Total Time: {no_div_metrics['avg_total_time']:.3f} s")

## Create Pruned Model (With Divisor=256)

Prune to 260% expansion rate WITH `expansion_divisor=256`.

In [ ]:
print("\n" + "="*60)
print("Creating pruned model WITH expansion_divisor=256")
print("="*60)

# Clean up and reload model
del pruned_no_divisor
cleanup_memory()

model_with_divisor = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

# Prune with divisor
pruned_with_divisor, stats_with_divisor = prune_model(
    model=model_with_divisor,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",
    pruning_percentage=None,  # Must be None when using expansion_rate
    expansion_rate=TARGET_EXPANSION_RATE,
    expansion_divisor=256,  # Round to multiple of 256
    show_progress=True,
    return_stats=True
)

# Model info
with_div_params = count_parameters(pruned_with_divisor)
with_div_intermediate = get_intermediate_size(pruned_with_divisor)

print(f"\nPruned Model (Divisor=256) Information:")
print(f"  Total parameters: {with_div_params:,}")
print(f"  Intermediate size: {with_div_intermediate:,}")
print(f"  Expansion rate: {stats_with_divisor['expansion_rate']:.2f}%")
print(f"  Parameter reduction: {stats_with_divisor['percentage_reduction']:.2f}%")
print(f"  Divisible by 256: {with_div_intermediate % 256 == 0}")

print("\n" + "="*60)
print("Benchmarking pruned model (divisor=256)...")
print("="*60)

with_div_metrics = benchmark_model(
    pruned_with_divisor, 
    tokenizer, 
    TEST_PROMPTS,
    max_new_tokens=MAX_NEW_TOKENS,
    num_runs=NUM_MEASURED_RUNS,
    warmup_runs=NUM_WARMUP_RUNS
)

print(f"\nPruned Model (Divisor=256) Performance:")
print(f"  Avg TTFT: {with_div_metrics['avg_ttft']*1000:.2f} ms (±{with_div_metrics['ttft_std']*1000:.2f})")
print(f"  Avg Tokens/Second: {with_div_metrics['avg_tokens_per_second']:.2f} (±{with_div_metrics['tps_std']:.2f})")
print(f"  Avg Total Time: {with_div_metrics['avg_total_time']:.3f} s")

# Clean up
del pruned_with_divisor
cleanup_memory()

## Results Comparison

Let's create a comprehensive comparison of all three models.

In [ ]:
# Reload base model for final comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

# Create comparison dataframe
comparison_data = {
    'Model': ['Base Model', 'Pruned (No Divisor)', 'Pruned (Divisor=256)'],
    'Parameters': [base_params, no_div_params, with_div_params],
    'Intermediate Size': [base_intermediate, no_div_intermediate, with_div_intermediate],
    'Expansion Rate (%)': [
        (base_intermediate/base_hidden)*100,
        stats_no_divisor['expansion_rate'],
        stats_with_divisor['expansion_rate']
    ],
    'Param Reduction (%)': [0.0, stats_no_divisor['percentage_reduction'], stats_with_divisor['percentage_reduction']],
    'Divisible by 256': [base_intermediate % 256 == 0, no_div_intermediate % 256 == 0, with_div_intermediate % 256 == 0],
    'TTFT (ms)': [
        base_metrics['avg_ttft']*1000,
        no_div_metrics['avg_ttft']*1000,
        with_div_metrics['avg_ttft']*1000
    ],
    'Tokens/Second': [
        base_metrics['avg_tokens_per_second'],
        no_div_metrics['avg_tokens_per_second'],
        with_div_metrics['avg_tokens_per_second']
    ],
}

df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*80)
print(f"\n{df.to_string(index=False)}\n")

# Calculate relative improvements
print("\nPerformance Improvements vs Base Model:")
print("-" * 80)

for i, model_name in enumerate(['Pruned (No Divisor)', 'Pruned (Divisor=256)']):
    ttft_improvement = ((base_metrics['avg_ttft'] - [no_div_metrics, with_div_metrics][i]['avg_ttft']) / base_metrics['avg_ttft']) * 100
    tps_improvement = (([no_div_metrics, with_div_metrics][i]['avg_tokens_per_second'] - base_metrics['avg_tokens_per_second']) / base_metrics['avg_tokens_per_second']) * 100
    
    print(f"\n{model_name}:")
    print(f"  TTFT improvement: {ttft_improvement:+.2f}%")
    print(f"  Tokens/Second improvement: {tps_improvement:+.2f}%")

print("\n" + "="*80)
print("\nDirect Comparison: No Divisor vs Divisor=256")
print("-" * 80)

ttft_diff = ((with_div_metrics['avg_ttft'] - no_div_metrics['avg_ttft']) / no_div_metrics['avg_ttft']) * 100
tps_diff = ((with_div_metrics['avg_tokens_per_second'] - no_div_metrics['avg_tokens_per_second']) / no_div_metrics['avg_tokens_per_second']) * 100
param_diff = with_div_params - no_div_params
intermediate_diff = with_div_intermediate - no_div_intermediate

print(f"Intermediate size difference: {intermediate_diff:+d} neurons")
print(f"Parameter difference: {param_diff:+,} ({((param_diff/no_div_params)*100):+.2f}%)")
print(f"TTFT difference: {ttft_diff:+.2f}%")
print(f"Tokens/Second difference: {tps_diff:+.2f}%")

## Visualizations

Let's create visual comparisons of the performance metrics.

In [ ]:
# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Performance Comparison: expansion_divisor Impact', fontsize=16, fontweight='bold')

# 1. TTFT Comparison (Bar Chart)
ax1 = axes[0, 0]
models = df['Model'].tolist()
ttft_values = df['TTFT (ms)'].tolist()
colors = ['#3498db', '#e74c3c', '#2ecc71']

bars1 = ax1.bar(models, ttft_values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('TTFT (milliseconds)', fontsize=12)
ax1.set_title('Time To First Token (Lower is Better)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=15)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}ms',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2. Tokens/Second Comparison (Bar Chart)
ax2 = axes[0, 1]
tps_values = df['Tokens/Second'].tolist()

bars2 = ax2.bar(models, tps_values, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Tokens per Second', fontsize=12)
ax2.set_title('Generation Throughput (Higher is Better)', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=15)

# Add value labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Parameter Count Comparison (Bar Chart)
ax3 = axes[1, 0]
param_values = [p/1e6 for p in df['Parameters'].tolist()]  # Convert to millions

bars3 = ax3.bar(models, param_values, color=colors, alpha=0.7, edgecolor='black')
ax3.set_ylabel('Parameters (Millions)', fontsize=12)
ax3.set_title('Model Size Comparison', fontsize=13, fontweight='bold')
ax3.tick_params(axis='x', rotation=15)

# Add value labels on bars
for bar, reduction in zip(bars3, df['Param Reduction (%)'].tolist()):
    height = bar.get_height()
    label = f'{height:.1f}M'
    if reduction > 0:
        label += f'\n({reduction:.1f}% reduction)'
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            label,
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Intermediate Size Comparison (Bar Chart)
ax4 = axes[1, 1]
intermediate_values = df['Intermediate Size'].tolist()

bars4 = ax4.bar(models, intermediate_values, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('Intermediate Layer Size', fontsize=12)
ax4.set_title('MLP Intermediate Dimension', fontsize=13, fontweight='bold')
ax4.tick_params(axis='x', rotation=15)

# Add value labels on bars with divisibility info
for bar, divisible in zip(bars4, df['Divisible by 256'].tolist()):
    height = bar.get_height()
    label = f'{int(height):,}'
    if divisible:
        label += '\n✓ Div by 256'
    else:
        label += '\n✗ Not div by 256'
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            label,
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('performance_comparison_expansion_divisor.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualizations created and saved!")

## Key Findings and Recommendations

Let's summarize the key findings from our performance comparison.

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Calculate key metrics
base_to_no_div_speedup = no_div_metrics['avg_tokens_per_second'] / base_metrics['avg_tokens_per_second']
base_to_with_div_speedup = with_div_metrics['avg_tokens_per_second'] / base_metrics['avg_tokens_per_second']
no_div_to_with_div_speedup = with_div_metrics['avg_tokens_per_second'] / no_div_metrics['avg_tokens_per_second']

print(f"\n1. MODEL SIZE REDUCTION")
print(f"   Both pruned models achieved similar size reductions:")
print(f"   - No divisor: {stats_no_divisor['percentage_reduction']:.2f}% reduction")
print(f"   - With divisor=256: {stats_with_divisor['percentage_reduction']:.2f}% reduction")
print(f"   - Difference: {abs(stats_no_divisor['percentage_reduction'] - stats_with_divisor['percentage_reduction']):.2f}%")

print(f"\n2. INTERMEDIATE LAYER SIZE")
print(f"   - Base: {base_intermediate:,} neurons")
print(f"   - No divisor: {no_div_intermediate:,} neurons (not aligned)")
print(f"   - With divisor=256: {with_div_intermediate:,} neurons (✓ aligned to 256)")
print(f"   - Difference: {abs(no_div_intermediate - with_div_intermediate)} neurons")

print(f"\n3. PERFORMANCE IMPROVEMENTS")
print(f"   Speedup vs Base Model:")
print(f"   - No divisor: {base_to_no_div_speedup:.2f}x faster")
print(f"   - With divisor=256: {base_to_with_div_speedup:.2f}x faster")
print(f"\n   Direct comparison (divisor=256 vs no divisor):")
print(f"   - Speedup: {no_div_to_with_div_speedup:.2f}x")
print(f"   - TTFT difference: {ttft_diff:+.2f}%")
print(f"   - Tokens/second difference: {tps_diff:+.2f}%")

print(f"\n4. HARDWARE OPTIMIZATION")
if with_div_intermediate % 256 == 0 and no_div_intermediate % 256 != 0:
    print(f"   ✓ expansion_divisor=256 ensures optimal GPU memory alignment")
    print(f"   ✓ Better utilization of tensor cores and SIMD instructions")
else:
    print(f"   Note: Both models may benefit from alignment on different hardware")

print("\n" + "="*80)
print("RECOMMENDATIONS")
print("="*80)

print(f"\n1. For Maximum Performance:")
print(f"   - Use expansion_divisor=256 on modern NVIDIA GPUs (A100, H100, RTX series)")
print(f"   - Ensures optimal memory access patterns and tensor core utilization")

print(f"\n2. When to Skip expansion_divisor:")
print(f"   - When the performance difference is negligible for your use case")
print(f"   - When model size is the primary concern over inference speed")
print(f"   - When targeting CPUs or specific hardware without alignment requirements")

print(f"\n3. Best Practices:")
print(f"   - Always benchmark on your target hardware")
print(f"   - Test different divisor values (32, 64, 128, 256) for optimal results")
print(f"   - Consider the trade-off between alignment and exact pruning ratio")

print("\n" + "="*80)

## Save Results

Let's save our comparison results to a CSV file for future reference.

In [ ]:
# Save comparison dataframe
df.to_csv('performance_comparison_results.csv', index=False)
print("✓ Results saved to 'performance_comparison_results.csv'")

# Save detailed metrics
detailed_results = {
    'base_model': {
        'parameters': base_params,
        'intermediate_size': base_intermediate,
        'ttft_ms': base_metrics['avg_ttft'] * 1000,
        'ttft_std_ms': base_metrics['ttft_std'] * 1000,
        'tokens_per_second': base_metrics['avg_tokens_per_second'],
        'tps_std': base_metrics['tps_std'],
    },
    'pruned_no_divisor': {
        'parameters': no_div_params,
        'intermediate_size': no_div_intermediate,
        'expansion_rate': stats_no_divisor['expansion_rate'],
        'param_reduction': stats_no_divisor['percentage_reduction'],
        'ttft_ms': no_div_metrics['avg_ttft'] * 1000,
        'ttft_std_ms': no_div_metrics['ttft_std'] * 1000,
        'tokens_per_second': no_div_metrics['avg_tokens_per_second'],
        'tps_std': no_div_metrics['tps_std'],
    },
    'pruned_with_divisor_256': {
        'parameters': with_div_params,
        'intermediate_size': with_div_intermediate,
        'expansion_rate': stats_with_divisor['expansion_rate'],
        'param_reduction': stats_with_divisor['percentage_reduction'],
        'ttft_ms': with_div_metrics['avg_ttft'] * 1000,
        'ttft_std_ms': with_div_metrics['ttft_std'] * 1000,
        'tokens_per_second': with_div_metrics['avg_tokens_per_second'],
        'tps_std': with_div_metrics['tps_std'],
    }
}

import json
with open('detailed_performance_metrics.json', 'w') as f:
    json.dump(detailed_results, f, indent=2)
print("✓ Detailed metrics saved to 'detailed_performance_metrics.json'")

print("\n" + "="*80)
print("All results saved successfully!")
print("="*80)

---
## ✅ Analysis Complete!

You've successfully compared the performance impact of using `expansion_divisor` when pruning models.

### Key Takeaways:

1. **Hardware Alignment Matters**: Using `expansion_divisor=256` ensures intermediate layer sizes are multiples of 256, which can improve GPU performance

2. **Minimal Size Impact**: The rounding typically changes the intermediate size by a small amount while maintaining similar overall parameter reduction

3. **Benchmark Your Hardware**: Performance improvements vary by hardware - always test on your target deployment platform

4. **Easy to Use**: Adding `expansion_divisor=256` is a one-parameter change that can provide measurable performance benefits

### Next Steps:

- Test with different models (larger/smaller)
- Try different divisor values (32, 64, 128)
- Benchmark on your specific hardware
- Combine with quantization for additional speedups

---

If you found this notebook useful, please ⭐ the [OptiPFair repository](https://github.com/peremartra/optipfair)!

### Follow the Project:

* **[LinkedIn](https://www.linkedin.com/in/pere-martra/)**
* **[X / Twitter](https://twitter.com/PereMartra)**
* **[GitHub](https://github.com/peremartra)**